# Bab 4 — Dataset Selection and Research Data Preparation

Notebook pendamping Bab 4 Level 7. Jalankan sel dari atas ke bawah. Bagian A sampai H mengikuti halaman bab. Bagian **Praktik** berisi kegiatan yang tidak ada di halaman: menambah kriteria eksklusimu sendiri, mengisi kartu dokumentasi untuk dataset lain, dan memeriksa keterwakilan.

Dataset: Breast Cancer Wisconsin (Diagnostic), UCI, CC BY 4.0 menurut halaman UCI. Bukan alat diagnosis.

## Learning Objectives

Setelah mengerjakan notebook ini, kamu mampu:

- Menulis kriteria eksklusi sebagai kode dan melihat dampaknya pada jumlah baris dan sidik jari data.
- Mengisi kartu dokumentasi dan menemukan bagian yang masih tidak diketahui.
- Memeriksa keterwakilan dengan membandingkan proporsi kelas dengan sumber yang kamu verifikasi sendiri.

## Langkah 0: Pindah ke folder utama proyek

In [1]:
import os
import sys
from pathlib import Path

if not Path("data").exists() and not Path("examples").exists():
    os.chdir("../..")

ROOT = Path.cwd()
# alat bantu bersama Level 7 (examples/level-7/_research.py)
sys.path.insert(0, str(Path("examples/level-7").resolve()))

## Bagian A: Siapkan alat

In [2]:
import json

import numpy as np
import pandas as pd

import _research as rs

X_all, diagnosis = rs.load_wdbc()
RESULTS = rs.RESULTS_DIR
print("Dimuat:", X_all.shape, "| label:", len(diagnosis))

Dimuat: (569, 30) | label: 569


## Bagian B: Kartu dokumentasi dataset

In [3]:
UNKNOWN = "tidak dinyatakan pada sumber yang dibaca"
card = {
    "nama": "Breast Cancer Wisconsin (Diagnostic)",
    "sumber_asli": "UCI Machine Learning Repository, https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic",
    "doi": "10.24432/C5DW2B",
    "pembuat": "William Wolberg, Olvi Mangasarian, Nick Street, W. Street (menurut halaman UCI)",
    "tanggal_donasi": "1995-10-31 (menurut halaman UCI)",
    "lisensi": "CC BY 4.0 (menurut halaman UCI); pemakaian wajib mencantumkan atribusi",
    "salinan_yang_dipakai": "sklearn.datasets.load_breast_cancer (salinan UCI yang disertakan scikit-learn)",
    "cara_pengumpulan": "Fitur dihitung dari citra digital aspirasi jarum halus (FNA) massa payudara; menggambarkan inti sel (menurut UCI dan scikit-learn)",
    "fitur": "10 ukuran inti sel, masing-masing dalam bentuk mean, standard error, dan worst = 30 fitur (menurut scikit-learn)",
    "label": "Diagnosis malignant atau benign (menurut UCI dan scikit-learn)",
    "tugas_asli": "Classification (menurut halaman UCI)",
    "persetujuan_etika_deidentifikasi": UNKNOWN,
    "demografi_pasien": UNKNOWN,
    "satuan_fitur": UNKNOWN,
    "makalah_pengantar": "Street, Wolberg, & Mangasarian (1993), Nuclear feature extraction for breast tumor diagnosis",
}
for key, value in card.items():
    print(f"{key:34s}: {value}")

nama                              : Breast Cancer Wisconsin (Diagnostic)
sumber_asli                       : UCI Machine Learning Repository, https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic
doi                               : 10.24432/C5DW2B
pembuat                           : William Wolberg, Olvi Mangasarian, Nick Street, W. Street (menurut halaman UCI)
tanggal_donasi                    : 1995-10-31 (menurut halaman UCI)
lisensi                           : CC BY 4.0 (menurut halaman UCI); pemakaian wajib mencantumkan atribusi
salinan_yang_dipakai              : sklearn.datasets.load_breast_cancer (salinan UCI yang disertakan scikit-learn)
cara_pengumpulan                  : Fitur dihitung dari citra digital aspirasi jarum halus (FNA) massa payudara; menggambarkan inti sel (menurut UCI dan scikit-learn)
fitur                             : 10 ukuran inti sel, masing-masing dalam bentuk mean, standard error, dan worst = 30 fitur (menurut scikit-learn)
label    

## Bagian C: Mencocokkan dokumentasi dengan data

In [4]:
# Angka dokumentasi (dari deskripsi scikit-learn dan halaman UCI) dibandingkan dengan data yang dimuat.
documented = {
    "jumlah_sampel": 569,
    "jumlah_fitur": 30,
    "nilai_hilang": 0,
    "malignant": 212,
    "benign": 357,
    "mean radius min": 6.981,
    "mean radius max": 28.11,
    "mean area min": 143.5,
    "mean area max": 2501.0,
}
counts = diagnosis.value_counts()
observed = {
    "jumlah_sampel": X_all.shape[0],
    "jumlah_fitur": X_all.shape[1],
    "nilai_hilang": int(X_all.isna().sum().sum()),
    "malignant": int(counts["malignant"]),
    "benign": int(counts["benign"]),
    "mean radius min": round(float(X_all["mean radius"].min()), 3),
    "mean radius max": round(float(X_all["mean radius"].max()), 3),
    "mean area min": round(float(X_all["mean area"].min()), 1),
    "mean area max": round(float(X_all["mean area"].max()), 1),
}
check = pd.DataFrame({"dokumentasi": documented, "data": observed}, dtype=object)
check["status"] = np.where(check["dokumentasi"] == check["data"], "cocok", "BERBEDA")
print(check.to_string())
print("Semua cocok:", bool((check["status"] == "cocok").all()))

                dokumentasi    data status
jumlah_sampel         569.0   569.0  cocok
jumlah_fitur           30.0    30.0  cocok
nilai_hilang            0.0     0.0  cocok
malignant             212.0   212.0  cocok
benign                357.0   357.0  cocok
mean radius min       6.981   6.981  cocok
mean radius max       28.11   28.11  cocok
mean area min         143.5   143.5  cocok
mean area max        2501.0  2501.0  cocok
Semua cocok: True


## Bagian D: Pemeriksaan kualitas

In [5]:
corr = X_all.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
pairs = upper.stack().sort_values(ascending=False)
q1, q3 = X_all.quantile(0.25), X_all.quantile(0.75)
iqr = q3 - q1
outlier_count = ((X_all < q1 - 1.5 * iqr) | (X_all > q3 + 1.5 * iqr)).sum().sort_values(ascending=False)
quality = {
    "baris_duplikat": int(X_all.duplicated().sum()),
    "kolom_konstan": int((X_all.nunique() <= 1).sum()),
    "nilai_negatif": int((X_all < 0).sum().sum()),
    "pasangan_fitur_korelasi_di_atas_0.95": int((pairs > 0.95).sum()),
    "fitur_miring_kuat_(|skew|>1)": int((X_all.skew().abs() > 1).sum()),
    "total_nilai_di_luar_pagar_IQR": int(outlier_count.sum()),
}
for key, value in quality.items():
    print(f"{key:40s}: {value}")
print("Lima pasangan fitur paling berkorelasi:")
print(pairs.head(5).round(3).to_string())
print("Lima fitur dengan nilai luar pagar IQR terbanyak:")
print(outlier_count.head(5).to_string())

baris_duplikat                          : 0
kolom_konstan                           : 0
nilai_negatif                           : 0
pasangan_fitur_korelasi_di_atas_0.95    : 15
fitur_miring_kuat_(|skew|>1)            : 22
total_nilai_di_luar_pagar_IQR           : 608
Lima pasangan fitur paling berkorelasi:
mean radius     mean perimeter     0.998
worst radius    worst perimeter    0.994
mean radius     mean area          0.987
mean perimeter  mean area          0.987
worst radius    worst area         0.984
Lima fitur dengan nilai luar pagar IQR terbanyak:
area error          65
radius error        38
perimeter error     38
worst area          35
smoothness error    30


## Bagian E: Kriteria inklusi dan eksklusi

In [6]:
# Kriteria ditulis sebelum menerapkannya. Setiap aturan mengembalikan baris yang LOLOS.
positive_by_definition = [c for c in X_all.columns if any(k in c for k in ("radius", "texture", "perimeter", "area"))]
rules = [
    ("I1 tidak ada nilai hilang pada 30 fitur", lambda f, d: ~f.isna().any(axis=1)),
    ("I2 punya label diagnosis (hanya validasi)", lambda f, d: d.notna()),
    ("E1 bukan baris duplikat persis", lambda f, d: ~f.duplicated(keep="first")),
    ("E2 fitur yang harus positif bernilai > 0", lambda f, d: (f[positive_by_definition] > 0).all(axis=1)),
]
keep = pd.Series(True, index=X_all.index)
funnel = [("Data awal", int(keep.sum()), 0)]
for name, rule in rules:
    passed = rule(X_all, diagnosis)
    removed = int((keep & ~passed).sum())
    keep &= passed
    funnel.append((name, int(keep.sum()), removed))
funnel_table = pd.DataFrame(funnel, columns=["langkah", "sisa_baris", "dikeluarkan"])
print(funnel_table.to_string(index=False))

                                  langkah  sisa_baris  dikeluarkan
                                Data awal         569            0
  I1 tidak ada nilai hilang pada 30 fitur         569            0
I2 punya label diagnosis (hanya validasi)         569            0
           E1 bukan baris duplikat persis         569            0
 E2 fitur yang harus positif bernilai > 0         569            0


## Bagian F: Daftar periksa etika dan privasi

In [7]:
ethics = pd.DataFrame(
    [
        ["Kategori data sensitif (kesehatan)", "Ya", "Subjek halaman UCI: Health and Medicine"],
        ["Kolom identitas pada salinan yang dipakai", "Tidak ada" if not any(c.lower() in ("id", "patient_id") for c in X_all.columns) else "ADA", "Diperiksa dengan kode pada nama kolom"],
        ["Persetujuan pasien dan tinjauan etik", "Tidak diketahui", "Halaman UCI tidak membahasnya"],
        ["Status de-identifikasi", "Tidak diketahui", "Halaman UCI tidak membahasnya"],
        ["Lisensi mengizinkan pemakaian ulang", "Ya, dengan atribusi", "CC BY 4.0 menurut halaman UCI"],
        ["Tujuan asli sama dengan tujuan kita", "Tidak sama", "Asli: classification; kita: metodologi clustering"],
        ["Klaim klinis dilakukan", "Tidak", "Dinyatakan di luar lingkup (Bab 2)"],
    ],
    columns=["pemeriksaan", "status", "dasar"],
)
print(ethics.to_string(index=False))
print("Butir 'Tidak diketahui':", int((ethics["status"] == "Tidak diketahui").sum()), "dari", len(ethics))

                              pemeriksaan              status                                             dasar
       Kategori data sensitif (kesehatan)                  Ya           Subjek halaman UCI: Health and Medicine
Kolom identitas pada salinan yang dipakai           Tidak ada             Diperiksa dengan kode pada nama kolom
     Persetujuan pasien dan tinjauan etik     Tidak diketahui                     Halaman UCI tidak membahasnya
                   Status de-identifikasi     Tidak diketahui                     Halaman UCI tidak membahasnya
      Lisensi mengizinkan pemakaian ulang Ya, dengan atribusi                     CC BY 4.0 menurut halaman UCI
      Tujuan asli sama dengan tujuan kita          Tidak sama Asli: classification; kita: metodologi clustering
                   Klaim klinis dilakukan               Tidak                Dinyatakan di luar lingkup (Bab 2)
Butir 'Tidak diketahui': 2 dari 7


## Bagian G: Dataset penelitian final

In [8]:
final_features = X_all.loc[keep].reset_index(drop=True)
final_diagnosis = diagnosis.loc[keep].reset_index(drop=True)
print("Fitur final   :", final_features.shape)
print("Label final   :", final_diagnosis.value_counts().to_dict())
print("Sidik jari awal :", rs.dataset_fingerprint(X_all))
print("Sidik jari final:", rs.dataset_fingerprint(final_features))
print("Data berubah    :", rs.dataset_fingerprint(X_all) != rs.dataset_fingerprint(final_features))

Fitur final   : (569, 30)
Label final   : {'benign': 357, 'malignant': 212}
Sidik jari awal : 6b202a2072f9
Sidik jari final: 6b202a2072f9
Data berubah    : False


## Bagian H: Menyimpan kartu, laporan, dan dataset

In [9]:
RESULTS.mkdir(exist_ok=True)
card["sidik_jari_data_final"] = rs.dataset_fingerprint(final_features)
card["jumlah_sampel_final"] = int(final_features.shape[0])
card["baris_dikeluarkan"] = int(X_all.shape[0] - final_features.shape[0])
card["kriteria"] = [name for name, _ in rules]
(RESULTS / "04_dataset_card.json").write_text(json.dumps(card, indent=2, ensure_ascii=False), encoding="utf-8")
check.reset_index(names="item").to_csv(RESULTS / "04_documentation_check.csv", index=False, encoding="utf-8")
funnel_table.to_csv(RESULTS / "04_inclusion_funnel.csv", index=False, encoding="utf-8")
ethics.to_csv(RESULTS / "04_ethics_checklist.csv", index=False, encoding="utf-8")
research_dataset = final_features.assign(diagnosis=final_diagnosis)
research_dataset.to_csv(RESULTS / "04_research_dataset.csv", index=False, encoding="utf-8")
for name in ["04_dataset_card.json", "04_documentation_check.csv", "04_inclusion_funnel.csv", "04_ethics_checklist.csv", "04_research_dataset.csv"]:
    print("Disimpan:", f"examples/level-7/results/{name}")

Disimpan: examples/level-7/results/04_dataset_card.json
Disimpan: examples/level-7/results/04_documentation_check.csv
Disimpan: examples/level-7/results/04_inclusion_funnel.csv
Disimpan: examples/level-7/results/04_ethics_checklist.csv
Disimpan: examples/level-7/results/04_research_dataset.csv


## Praktik 1: Tambahkan kriteria eksklusimu sendiri

Aturan di bawah mengeluarkan baris dengan `worst area` di atas ambang. Ubah ambangnya atau ganti dengan aturanmu sendiri, lalu lihat berapa baris yang keluar dan apakah sidik jari data berubah. **Ingat:** menambah aturan setelah melihat data harus dicatat dan dijustifikasi, bukan dipakai agar kelompok tampak lebih rapi.

In [10]:
my_rule_name = "E3 worst area <= 4000"
my_rule = lambda f, d: f["worst area"] <= 4000

keep2 = keep.copy()
passed = my_rule(X_all, diagnosis)
removed = X_all.index[keep2 & ~passed]
keep2 &= passed
print(my_rule_name, "-> dikeluarkan:", len(removed), "baris")
print("Kelas yang dikeluarkan:", diagnosis.loc[removed].value_counts().to_dict())
print("Baris tersisa:", int(keep2.sum()))
print("Sidik jari berubah:", rs.dataset_fingerprint(X_all.loc[keep2]) != rs.dataset_fingerprint(X_all))

E3 worst area <= 4000 -> dikeluarkan: 1 baris
Kelas yang dikeluarkan: {'malignant': 1}
Baris tersisa: 568
Sidik jari berubah: True


## Praktik 2: Kartu dokumentasi untuk dataset lain

Pilih dataset lain (misalnya yang kamu pakai untuk tugasmu), lalu ganti setiap `ISI` dengan fakta yang kamu verifikasi di sumber asli. Jika sumber tidak menyatakannya, tulis `UNKNOWN`. Fungsi `card_gaps` melaporkan bagian yang belum terisi dan bagian yang memang tidak diketahui.

In [11]:
my_card = {
    "nama": "ISI", "sumber_asli": "ISI", "doi_atau_url": "ISI",
    "pembuat": "ISI", "lisensi": "ISI",
    "cara_pengumpulan": "ISI", "jumlah_sampel": "ISI",
    "persetujuan_etika_deidentifikasi": "ISI",
    "tujuan_asli": "ISI",
}

def card_gaps(card):
    todo = [k for k, v in card.items() if str(v).strip() == "ISI"]
    unknown = [k for k, v in card.items() if v == UNKNOWN]
    return todo, unknown

todo, unknown = card_gaps(my_card)
print("Belum diisi :", len(todo), "dari", len(my_card))
print("Tidak diketahui:", unknown)
print("Kartu siap dipakai:", not todo)

Belum diisi : 9 dari 9
Tidak diketahui: []
Kartu siap dipakai: False


## Praktik 3: Periksa keterwakilan

Proporsi kelas pada data ini bisa dibandingkan dengan proporsi pada populasi yang ingin kamu bicarakan, **bila** kamu punya sumber terverifikasi untuk angka itu. Isi `assumed_prevalence` dengan angka dari sumber yang kamu baca (antara 0 dan 1). Selama masih `None`, sel hanya mencetak proporsi pada data dan mengingatkan bahwa pembandingnya belum ada.

In [12]:
share_malignant = float((diagnosis == "malignant").mean())
print("Proporsi malignant pada data:", round(share_malignant, 3))

assumed_prevalence = None   # isi dengan angka dari sumber yang kamu verifikasi, contoh: 0.05
if assumed_prevalence is None:
    print("Pembanding belum ada: keterwakilan terhadap populasi TIDAK bisa dinilai. Klaim populasi tidak boleh dibuat.")
else:
    print("Selisih proporsi (data - populasi):", round(share_malignant - assumed_prevalence, 3))

Proporsi malignant pada data: 0.373
Pembanding belum ada: keterwakilan terhadap populasi TIDAK bisa dinilai. Klaim populasi tidak boleh dibuat.


## Latihan

1. Ganti ambang pada Praktik 1 menjadi 3000. Berapa baris yang keluar, dan apa dampaknya pada kelas *malignant*? Apakah itu sesuai dengan keputusan "tidak membuang outlier" di bab?
2. Isi `my_card` untuk dataset lain sampai `card_gaps` melaporkan 0 yang belum diisi. Bagian mana yang berakhir `UNKNOWN`?
3. Cari satu sumber terverifikasi untuk proporsi kelas pada populasi yang relevan, isi `assumed_prevalence`, lalu tulis satu kalimat tentang seberapa jauh data ini mewakili populasi itu.
4. Tambahkan satu butir pada daftar periksa etika (Bagian F) yang menurutmu kurang. Apa statusnya untuk dataset ini?

## Ringkasan

- Kriteria inklusi dan eksklusi adalah kode yang bisa diulang; menambah aturan setelah melihat data harus dicatat dan dijustifikasi.
- Kartu dokumentasi yang jujur menandai hal yang **tidak diketahui**, bukan menebaknya.
- Keterwakilan tidak bisa dinilai tanpa pembanding yang terverifikasi; tanpa itu, klaim populasi tidak boleh dibuat.
- Dataset final disimpan apa adanya dengan sidik jari, dan penyamaan skala menjadi bagian protokol (Bab 5).